###R1

In [0]:
%py
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")

In [0]:
df = spark.read.csv(
    "/Volumes/claims/raw/landing/raw/members.csv", header=True, inferSchema=True
)

df.write.mode("overwrite").saveAsTable(f"{catalog}.raw.members")

In [0]:
df = spark.read.csv("/Volumes/claims/raw/landing/raw/hospital_network_history.csv", header=True, inferSchema=True)

df.write.mode("overwrite").saveAsTable(f"{catalog}.raw.hospital_network_history")

###R2

In [0]:
from pyspark.sql.functions import current_timestamp, col

source_path = f"/Volumes/{catalog}/raw/landing/raw/claims_raw/"
checkpoint_path = f"/Volumes/{catalog}/raw/landing/raw/"

claims_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/claims_checkpoints")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

bronze_claims_df = claims_stream.select(
    "*",
    col("_metadata.file_path").alias("_source_file"),
    current_timestamp().alias("_ingest_ts")
)

query = (
    bronze_claims_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(f"{catalog}.bronze.claims")
)